# Teacher cost (H3) and preset scores (GPU, one T4)

Two jobs in one session:

1. **Preset scores:** raw Qwen's six logits for the demo app's presets
   (`Deployment/frontend/presets.json`, written for the demo; none comes from a dataset), with
   labeling's own code (`label_worker.load_scorer`, `score_texts`). The TF-IDF half of the
   stack and the stack itself are applied afterwards on a CPU, from `presets_qwen.json`.
2. **Teacher serving cost:** vLLM's OpenAI-compatible server with labeling's engine settings,
   one request per comment (its six P2 prompts), load-tested on `localhost` at a sweep of
   request rates, then at saturation, then (if set) at the student load test's rate.
   Results in `h3_sweep.json`; `reports/h3_teacher_cost.json` is built from it.

**Kaggle setup:** accelerator GPU T4 (only GPU 0 is used, so the numbers are for one T4),
internet on, inputs: the Qwen weights dataset (`weights_manifest.json`) and the Civil drift
dataset (`civil_drift_ids.csv` and its batches), as for labeling. No Secrets: nothing is uploaded.

**Privacy:** the load-test comments never leave the session. The outputs hold ids, scores,
counts, latencies and length statistics only, and the server doesn't log requests (vLLM's
default). Clear this notebook's outputs before committing it.

In [ ]:
import time

SESSION_START = time.time()

REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
# The pinned commit: labeling's code (src/teacher) and the presets
# (Deployment/frontend/presets.json). 5daa102 is #60's head; any later main commit works too.
REPO_REF = "5daa102b0aa7f99dfb272f59acccd52a4398c268"
VLLM_VERSION = "0.31.0"     # as for labeling
GPU = "0"                   # one T4, even when the session has two
PORT = 8000

# Load test. Rates are comments per second (one request = one comment's six prompts).
N_TEXTS = 2000              # Civil drift comments sampled for the load (seeded)
WARMUP_S = 120
RATES = [0.5, 1, 2, 3, 4, 5, 6, 8]
RATE_S = 300                # per rate
SATURATION_CONCURRENCY = 64
SATURATION_S = 300
STABILITY_RATE = None       # the student load test's rate, if known; None skips the run
STABILITY_S = 1800
P95_TARGET_S = 2.0          # "keeps up": p95 at or under this, no errors, rate achieved
REQUEST_TIMEOUT_S = 60

In [ ]:
# Install vLLM before anything imports torch (as in label_shards.ipynb). 3 to 10 minutes.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", f"vllm=={VLLM_VERSION}"], check=True)
# The image's torchaudio is built for a different CUDA than vLLM's torch; nothing needs it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=True)

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("/tmp/Distillate")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
PIPELINE = REPO_DIR / "Model-Pipeline"
PRESETS_FILE = REPO_DIR / "Deployment" / "frontend" / "presets.json"
assert PRESETS_FILE.exists(), "REPO_REF has no Deployment/frontend/presets.json"
sys.path.insert(0, str(PIPELINE))
print("code_commit:", CODE_COMMIT)

In [ ]:
# One GPU, read with nvidia-smi so this kernel never initializes CUDA (the server needs it).
GPUS = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total",
     "--format=csv,noheader"], text=True).strip().splitlines()
print("\n".join(GPUS))
GPU_NAME = next(line.split(", ")[1] for line in GPUS if line.split(", ")[0] == GPU)
assert "T4" in GPU_NAME, f"expected a T4, got {GPU_NAME}"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU     # inherited by the scoring job and the server

In [ ]:
import json

from src.teacher.config import load_config, model_version
from src.teacher.data import clean_text, find_file
from src.teacher.prompts import prompt_version

cfg = load_config()
P = cfg["prompt"]
PROMPT_VERSION = prompt_version(P["protocol"], P["shots"], P["text_version"])
MODEL_VERSION = model_version(cfg)
MODEL_DIR = find_file("/kaggle/input", "weights_manifest.json").parent
weights_manifest = json.loads((MODEL_DIR / "weights_manifest.json").read_text())
assert weights_manifest["revision"] == cfg["model"]["revision"], weights_manifest["revision"]
DRIFT_IDS = find_file("/kaggle/input", cfg["drift"]["ids_file"])
PRESETS = json.loads(PRESETS_FILE.read_text(encoding="utf-8"))["presets"]
OUT = Path("/kaggle/working")
print(PROMPT_VERSION, MODEL_VERSION, f"{len(PRESETS)} presets")

In [ ]:
# Copy the weights to local disk once: reading them over Kaggle's network mount (GCS FUSE) is
# slow (vLLM took 24 minutes there on a T4), and both the preset job and the server load them.
import shutil

LOCAL_MODEL = Path("/tmp/qwen3-4b-instruct-2507")
if not (LOCAL_MODEL / "weights_manifest.json").exists():
    start = time.time()
    shutil.copytree(MODEL_DIR, LOCAL_MODEL, dirs_exist_ok=True)
    print(f"copied in {time.time() - start:.0f} s")
MODEL_DIR = LOCAL_MODEL

## 1. Preset scores (offline, labeling's code)

In its own process, so the GPU is free for the server afterwards. The presets are cleaned once
with `clean_text`, as the data lane cleans every comment before labeling.

In [ ]:
JOB_A = r"""
import json, sys
sys.path.insert(0, sys.argv[1])
from src.teacher.config import load_config, model_version
from src.teacher.data import clean_text
from src.teacher.label_worker import load_scorer
from src.teacher.prompts import PromptBuilder
from src.teacher.scoring import score_texts

cfg = load_config()
presets = json.loads(open(sys.argv[2], encoding="utf-8").read())["presets"]
tokenizer, score_fn, vllm_version = load_scorer(cfg, sys.argv[3])
builder = PromptBuilder.from_config(cfg, tokenizer)
frame, info = score_texts([p["id"] for p in presets], [clean_text(p["text"]) for p in presets],
                          builder, score_fn, cfg["scoring"]["min_coverage"],
                          cfg["scoring"]["max_retries"])
out = {"prompt_version": builder.prompt_version, "model_version": model_version(cfg),
       "vllm_version": vllm_version, "code_commit": sys.argv[5], "info": info,
       "rows": json.loads(frame.to_json(orient="records"))}
open(sys.argv[4], "w").write(json.dumps(out, indent=2))
"""
Path("/tmp/score_presets.py").write_text(JOB_A)
start = time.time()
subprocess.run([sys.executable, "/tmp/score_presets.py", str(PIPELINE), str(PRESETS_FILE),
                str(MODEL_DIR), str(OUT / "presets_qwen.json"), CODE_COMMIT], check=True)
PRESETS_QWEN = json.loads((OUT / "presets_qwen.json").read_text())
print(f"{time.time() - start:.0f} s", PRESETS_QWEN["info"])
assert PRESETS_QWEN["info"]["failed_rows"] == 0, "a preset failed: check coverage"
assert PRESETS_QWEN["prompt_version"] == PROMPT_VERSION

## 2. The teacher as a service

vLLM's OpenAI-compatible server with labeling's engine settings (`label_worker.load_scorer`).
Requests aren't logged (vLLM's default); the log goes to `/tmp`, outside the session's output.

In [ ]:
import requests

URL = f"http://127.0.0.1:{PORT}"
SERVER_LOG = Path("/tmp/vllm_server.log")
server = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.cli.main", "serve", str(MODEL_DIR),
     "--served-model-name", "teacher", "--host", "127.0.0.1", "--port", str(PORT),
     "--dtype", cfg["model"]["dtype"], "--max-model-len", str(cfg["model"]["max_model_len"]),
     "--gpu-memory-utilization", "0.90", "--enable-prefix-caching",
     "--generation-config", "vllm", "--seed", str(cfg["seed"]),
     "--max-logprobs", str(cfg["scoring"]["logprobs"]), "--disable-uvicorn-access-log"],
    stdout=SERVER_LOG.open("w"), stderr=subprocess.STDOUT)

start = time.time()
while True:
    if server.poll() is not None:
        print("\n".join(SERVER_LOG.read_text().splitlines()[-40:]))
        raise RuntimeError(f"the server exited with code {server.returncode}")
    try:
        if requests.get(f"{URL}/health", timeout=5).status_code == 200:
            break
    except requests.RequestException:
        pass
    if time.time() - start > 1200:
        raise RuntimeError("the server didn't become healthy in 20 minutes")
    time.sleep(10)
SERVER_READY_S = round(time.time() - start, 1)
print(f"server ready in {SERVER_READY_S} s")

In [ ]:
# The wrapper: one comment -> its six P2 prompts in one /v1/completions request -> six scores,
# read exactly as labeling reads them (scoring.yes_probability on the first token's top-k).
from transformers import AutoTokenizer

from src.teacher.config import LABELS
from src.teacher.prompts import PromptBuilder
from src.teacher.scoring import answer_token_ids, yes_probability

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
YES_IDS, NO_IDS = answer_token_ids(tokenizer)
builder = PromptBuilder.from_config(cfg, tokenizer)


def request_body(prompts):
    return {"model": "teacher", "prompt": list(prompts), "max_tokens": 1, "temperature": 0.0,
            "logprobs": cfg["scoring"]["logprobs"], "return_tokens_as_token_ids": True}


def parse(body):
    """Six Scores from a /v1/completions answer, in prompt (label) order."""
    choices = sorted(body["choices"], key=lambda c: c["index"])
    if len(choices) != len(LABELS):
        raise ValueError(f"expected {len(LABELS)} choices, got {len(choices)}")
    scores = []
    for choice in choices:
        top = choice["logprobs"]["top_logprobs"][0]
        ids = {int(token.split(":", 1)[1]): logprob for token, logprob in top.items()}
        scores.append(yes_probability(ids, YES_IDS, NO_IDS))
    return scores


def score_comment(text):
    prompts, _ = builder.build(text)
    response = requests.post(f"{URL}/v1/completions", json=request_body(prompts),
                             timeout=REQUEST_TIMEOUT_S)
    response.raise_for_status()
    return parse(response.json())


# Parity: the presets through the server must match the offline scores (fp16 batching noise
# only). A real difference means the server path isn't the labeling path, so stop.
offline = {row["id"]: row for row in PRESETS_QWEN["rows"]}
diff_p, diff_logit = [], []
for preset in PRESETS:
    served = score_comment(clean_text(preset["text"]))
    for label, score in zip(LABELS, served, strict=True):
        diff_p.append(abs(score.p - offline[preset["id"]][f"p_{label}"]))
        diff_logit.append(abs(score.logit - offline[preset["id"]][f"logit_{label}"]))
PARITY = {"cells": len(diff_p), "max_abs_p": max(diff_p), "max_abs_logit": max(diff_logit)}
print(PARITY)
assert PARITY["max_abs_p"] <= 0.02, "server scores differ from offline scores"

## 3. Load test

The load is Civil drift comments, the traffic the service sees, sampled with a fixed seed. Their
prompts are built once here; only length statistics are kept.

In [ ]:
import random
import statistics

from src.teacher.label_worker import select_drift_rows

rows = select_drift_rows(DRIFT_IDS.parent, DRIFT_IDS)
sample = rows.sample(n=min(N_TEXTS, len(rows)), random_state=cfg["seed"])["text"].tolist()
BODIES, truncated, prompt_tokens = [], 0, []
for text in sample:
    prompts, was_truncated = builder.build(text)
    truncated += was_truncated
    prompt_tokens.append(sum(len(tokenizer.encode(p, add_special_tokens=False))
                             for p in prompts))
    BODIES.append(request_body(prompts))
chars = sorted(len(t) for t in sample)


def pct(values, q):
    values = sorted(values)
    return values[min(len(values) - 1, int(q * len(values)))]


TEXTS = {"source": "Civil drift batches (civil_drift_ids.csv), seeded sample",
         "comments": len(sample), "truncated": truncated,
         "chars": {"mean": round(statistics.mean(chars), 1), "p50": pct(chars, 0.5),
                   "p95": pct(chars, 0.95)},
         "prompt_tokens_per_comment": {"mean": round(statistics.mean(prompt_tokens), 1),
                                       "p50": pct(prompt_tokens, 0.5),
                                       "p95": pct(prompt_tokens, 0.95)}}
del rows, sample   # the comments stay only inside the request bodies, in memory
print(TEXTS)

In [ ]:
import asyncio
import threading
from concurrent.futures import ThreadPoolExecutor

import aiohttp


class GpuMonitor:
    """GPU utilization and memory every 2 s, from nvidia-smi, while a run is going."""

    def __init__(self):
        self.samples, self._stop = [], threading.Event()

    def _loop(self):
        while not self._stop.is_set():
            try:   # a failed reading loses one sample, never the run
                out = subprocess.run(["nvidia-smi", "-i", GPU,
                                      "--query-gpu=utilization.gpu,memory.used",
                                      "--format=csv,noheader,nounits"],
                                     capture_output=True, text=True, timeout=10).stdout
                util, mem = (float(x) for x in out.strip().split(","))
                self.samples.append((util, mem))
            except (OSError, subprocess.SubprocessError, ValueError):
                pass
            self._stop.wait(2)

    def __enter__(self):
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()

    def summary(self):
        if not self.samples:
            return {}
        util = [s[0] for s in self.samples]
        return {"gpu_util_mean": round(statistics.mean(util), 1), "gpu_util_max": max(util),
                "gpu_mem_max_mib": max(s[1] for s in self.samples)}


async def post(session, body):
    sent = time.perf_counter()
    try:
        async with session.post(f"{URL}/v1/completions", json=body) as response:
            payload = await response.json()
            ok = response.status == 200 and len(payload.get("choices", [])) == len(LABELS)
    except (aiohttp.ClientError, asyncio.TimeoutError, ValueError):
        ok = False
    done = time.perf_counter()
    return ok, sent, done


def summarize(results, seconds, offered=None, **extra):
    ok = [(s, d) for good, s, d in results if good]
    latencies = sorted(d - s for s, d in ok)
    span = (max(d for _, d in ok) - min(s for s, _ in ok)) if ok else 0
    out = {"seconds": seconds, "sent": len(results), "ok": len(ok),
           "errors": len(results) - len(ok),
           "achieved_rate": round(len(ok) / span, 3) if span else 0.0, **extra}
    if offered is not None:
        out["offered_rate"] = offered
    if latencies:
        out.update({"latency_p50_s": round(pct(latencies, 0.5), 3),
                    "latency_p95_s": round(pct(latencies, 0.95), 3),
                    "latency_p99_s": round(pct(latencies, 0.99), 3),
                    "latency_mean_s": round(statistics.mean(latencies), 3)})
    out["keeps_up"] = bool(
        latencies and out["errors"] == 0 and out["latency_p95_s"] <= P95_TARGET_S
        and (offered is None or out["achieved_rate"] >= 0.95 * offered))
    return out


async def open_loop(rate, seconds, seed):
    """Poisson arrivals at ``rate`` comments/s for ``seconds``; every request is awaited."""
    rng = random.Random(seed)
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_S)
    async with aiohttp.ClientSession(timeout=timeout,
                                     connector=aiohttp.TCPConnector(limit=0)) as session:
        start, tasks, t, i = time.perf_counter(), [], 0.0, 0
        while True:
            t += rng.expovariate(rate)
            if t > seconds:
                break
            await asyncio.sleep(max(0.0, start + t - time.perf_counter()))
            tasks.append(asyncio.create_task(post(session, BODIES[(seed + i) % len(BODIES)])))
            i += 1
        return await asyncio.gather(*tasks)


async def closed_loop(concurrency, seconds):
    """``concurrency`` clients sending back to back for ``seconds``: the throughput ceiling."""
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_S)
    results, deadline = [], time.perf_counter() + seconds

    async def client(k, session):
        i = k
        while time.perf_counter() < deadline:
            results.append(await post(session, BODIES[i % len(BODIES)]))
            i += concurrency

    async with aiohttp.ClientSession(timeout=timeout,
                                     connector=aiohttp.TCPConnector(limit=0)) as session:
        await asyncio.gather(*(client(k, session) for k in range(concurrency)))
    return results


def run(coroutine_fn, *args):
    """Run an async load in its own thread and event loop (the notebook's loop stays free)."""
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(lambda: asyncio.run(coroutine_fn(*args))).result()

In [ ]:
RUNS = []
with GpuMonitor():
    run(open_loop, 1.0, WARMUP_S, 0)           # warm-up: compiles, fills the prefix cache
print("warm-up done")

for k, rate in enumerate(RATES, start=1):
    with GpuMonitor() as gpu:
        results = run(open_loop, rate, RATE_S, k)
    RUNS.append(summarize(results, RATE_S, offered=rate, kind="open_loop", **gpu.summary()))
    print(RUNS[-1])
    last = RUNS[-1]
    if last["achieved_rate"] < 0.8 * rate or last.get("latency_p95_s", 1e9) > 10 * P95_TARGET_S:
        print(f"saturated at {rate}/s: higher rates would only queue, so the sweep stops here")
        break

with GpuMonitor() as gpu:
    results = run(closed_loop, SATURATION_CONCURRENCY, SATURATION_S)
SATURATION = summarize(results, SATURATION_S, kind="closed_loop",
                       concurrency=SATURATION_CONCURRENCY, **gpu.summary())
print("saturation:", SATURATION)

STABILITY = None
if STABILITY_RATE:
    with GpuMonitor() as gpu:
        results = run(open_loop, STABILITY_RATE, STABILITY_S, 999)
    STABILITY = summarize(results, STABILITY_S, offered=STABILITY_RATE, kind="open_loop",
                          **gpu.summary())
    print("stability:", STABILITY)

In [ ]:
import vllm

kept = [r for r in RUNS if r["keeps_up"]]
SWEEP = {
    "what": "teacher (raw Qwen, P2 zero-shot) served by vLLM's OpenAI-compatible server on one "
            "GPU; one request = one comment's six prompts; load generator on localhost",
    "keeps_up_rule": f"p95 <= {P95_TARGET_S} s, no errors, achieved >= 95% of offered",
    "sustainable_rate": max((r["offered_rate"] for r in kept), default=None),
    "saturation": SATURATION, "stability": STABILITY, "runs": RUNS, "texts": TEXTS,
    "parity_with_offline": PARITY,
    "environment": {"gpu": GPU_NAME, "gpus_in_session": len(GPUS),
                    "vllm_version": vllm.__version__, "model_version": MODEL_VERSION,
                    "prompt_version": PROMPT_VERSION, "code_commit": CODE_COMMIT,
                    "dtype": cfg["model"]["dtype"],
                    "max_model_len": cfg["model"]["max_model_len"],
                    "gpu_memory_utilization": 0.90, "prefix_caching": True,
                    "logprobs": cfg["scoring"]["logprobs"],
                    "server_ready_s": SERVER_READY_S},
    "settings": {"warmup_s": WARMUP_S, "rate_s": RATE_S, "rates": RATES,
                 "saturation_concurrency": SATURATION_CONCURRENCY,
                 "saturation_s": SATURATION_S, "stability_rate": STABILITY_RATE,
                 "stability_s": STABILITY_S, "request_timeout_s": REQUEST_TIMEOUT_S,
                 "seed": cfg["seed"]},
    "session_hours": round((time.time() - SESSION_START) / 3600, 2),
}
(OUT / "h3_sweep.json").write_text(json.dumps(SWEEP, indent=2))
server.terminate()
server.wait(timeout=120)
print("sustainable rate:", SWEEP["sustainable_rate"], "comments/s | saturation:",
      SATURATION["achieved_rate"], "comments/s")

In [ ]:
# Download links, for an interactive session: its files are deleted when it stops. (A saved
# version keeps them under the notebook's Output tab instead.) Only these two files are needed.
from IPython.display import FileLink, display

os.chdir(OUT)   # FileLink paths are relative to the working directory
for name in ("presets_qwen.json", "h3_sweep.json"):
    if (OUT / name).exists():
        display(FileLink(name))
    else:
        print(f"{name} wasn't written: check the cells above")

## After the session (on a CPU, not here)

Download `presets_qwen.json` and `h3_sweep.json` from the output. Then:

* **Preset stack scores:** the full-train TF-IDF model (`baseline_tfidf.train_predict` on
  `wiki_train_raw`) scores the presets, and `stack.apply_stack` with `stack-v1`
  (`artifacts/stack/stack_p2-zs-v2b.json`) combines them with `presets_qwen.json`; the result
  fills `stack_scores` in `Deployment/frontend/presets.json`.
* **`reports/h3_teacher_cost.json`:** instances needed at the student's rate (its rate over
  `sustainable_rate`, rounded up), priced at GCP's list rate for a T4 plus its host machine
  (with the date and source), and cost per million comments. It states that the teacher was
  measured on one T4, on `localhost` (no network time), and that TF-IDF's cost is negligible.